# Challenge Motiva: Dynamic Programming (DP)
## Notebook de execução

Este notebook roda, passo a passo, as entregas do DP: a geração do dataset (Etapa 0), as buscas (Atividade 1), o percurso na linha com pilha (Atividade 2) e o plano do turno com fila e pilha (Atividade 3).

Cada etapa tem uma célula de explicação e uma célula de código. Rode as células em ordem.

**Antes de começar:** no Colab, envie para a pasta de trabalho os arquivos `gerar_dataset.py`, `dados.py`, `busca.py`, `linha.py`, `turno.py` e `linha.csv`. Para o Jupyter no seu computador, basta estar na pasta do projeto.

### Etapa 0: preparação e geração do dataset

Primeiro, verificamos se estamos no Colab e, se estivermos, se os arquivos do projeto já estão na pasta. Depois, geramos o dataset com a semente 42. Ela torna o sorteio reproduzível: quem rodar de novo obtém os mesmos números.

In [37]:
import os, sys

if "google.colab" in sys.modules:
    arquivos = ["gerar_dataset.py", "dados.py", "busca.py", "linha.py", "turno.py", "linha.csv"]
    faltando = [a for a in arquivos if not os.path.exists(a)]
    print("Arquivos que faltam no Colab:", faltando or "nenhum")

from gerar_dataset import gerar_dataset, resumir_dataset

leituras_geradas, escadas = gerar_dataset(n_escadas=50, n_leituras=200, semente=42, caminho_saida="sensores.csv")
resumo = resumir_dataset(leituras_geradas, escadas)

print("===== RESUMO POR SENSOR =====")
for sensor, valores in resumo["sensores"].items():
    print(f"{sensor:12} | min: {valores['minimo']:7.2f} | media: {valores['media']:7.2f} | max: {valores['maximo']:7.2f}")

print("\n===== ESCADAS POR PERFIL =====")
for perfil, quantidade in resumo["perfis"].items():
    print(f"{perfil:10}: {quantidade} escadas")

print(f"\nTotal de leituras: {resumo['total_leituras']}")


Arquivos que faltam no Colab: nenhum
===== RESUMO POR SENSOR =====
vibracao     | min:    1.00 | media:    3.13 | max:   10.00
temperatura  | min:   35.00 | media:   53.16 | max:   89.90
corrente     | min:   10.00 | media:   17.48 | max:   25.00
tensao       | min:  361.00 | media:  379.84 | max:  399.00
nivel_oleo   | min:   30.60 | media:   75.47 | max:  100.00
velocidade   | min:    0.48 | media:    0.50 | max:    0.52

===== ESCADAS POR PERFIL =====
saudavel  : 30 escadas
desgaste  : 15 escadas
falha     : 5 escadas

Total de leituras: 10000


**O que o resumo mostra:** o mínimo, a média e o máximo de cada sensor. Confira se os valores ficam dentro das faixas do enunciado (vibração 1 a 4 mm/s normalmente, temperatura 35 a 65 °C, e assim por diante). Os valores acima da faixa vêm do desgaste e das falhas, não de erro no código.

### Atividade 1: busca linear × busca binária

**Passo 1.** Lemos o `sensores.csv` com a função `ler_leituras`. Ela converte os sensores para número.

In [26]:
from dados import ler_leituras, esta_em_alerta
from busca import (busca_linear, busca_binaria, ordenar_ids, acumular,
                   primeira_leitura_linear, primeira_leitura_binaria)

leituras = ler_leituras("sensores.csv")
print("Leituras lidas:", len(leituras))
print("Primeira leitura:", leituras[0])

Leituras lidas: 10000
Primeira leitura: {'id_escada': 'ESC-10', 'estacao': 'Pinheiros', 'grupo': 'não crítica', 'data_hora': '2026-01-01 00:00', 'vibracao': 3.42, 'temperatura': 40.7, 'corrente': 11.45, 'tensao': 377.4, 'nivel_oleo': 76.9, 'velocidade': 0.499}


**Passo 2.** Agrupamos as leituras por escada, mantendo a ordem de horário. Também criamos a lista de IDs ordenada pelo número (`ESC-2` antes de `ESC-10`). A busca binária exige essa ordem.

In [27]:
por_escada = {}
for leitura in leituras:
    por_escada.setdefault(leitura["id_escada"], []).append(leitura)

ids_ordenados = ordenar_ids(por_escada.keys())
lista_ordenada = [{"id": i} for i in ids_ordenados]
print("Escadas:", len(ids_ordenados), "| primeiras:", ids_ordenados[:5])

Escadas: 50 | primeiras: ['ESC-1', 'ESC-2', 'ESC-3', 'ESC-4', 'ESC-5']


**Passo 3.** Fazemos 1.000 consultas por ID. Os IDs são sorteados com a semente 42, então todos os métodos buscam os mesmos IDs. Contamos as comparações de cada busca.

In [28]:
import random

rnd = random.Random(42)
consultas = [rnd.choice(ids_ordenados) for _ in range(1000)]

comp_linear_arquivo = [busca_linear(leituras, alvo, "id_escada")[1] for alvo in consultas]
comp_linear_ids = [busca_linear(lista_ordenada, alvo, "id")[1] for alvo in consultas]
comp_binaria = [busca_binaria(lista_ordenada, alvo, "id")[1] for alvo in consultas]

print(f"{'Método':30} | {'total':>7} | {'média':>6} | {'máx':>4}")
for nome, comps in [("Linear na lista do arquivo", comp_linear_arquivo),
                    ("Linear na lista de IDs", comp_linear_ids),
                    ("Binária na lista de IDs", comp_binaria)]:
    print(f"{nome:30} | {sum(comps):>7} | {sum(comps)/len(comps):>6.2f} | {max(comps):>4}")

Método                         |   total |  média |  máx
Linear na lista do arquivo     |   26305 |  26.30 |   50
Linear na lista de IDs         |   25835 |  25.84 |   50
Binária na lista de IDs        |    9342 |   9.34 |   12


**Leitura da tabela.** A busca binária faz muito menos comparações na mesma lista de IDs. A linear no arquivo tem média parecida com a linear na lista de IDs, porque o arquivo embaralha as escadas a cada hora. Assim, a primeira ocorrência de cada escada cai nas primeiras 50 linhas. Esse número depende do arquivo, não só do algoritmo.

**Passo 4.** Para cada escada, procuramos a primeira leitura em que os alertas acumulados chegam a 5. Usamos as duas buscas e conferimos se dão o mesmo resultado.

Uma leitura está em alerta quando algum sensor sai da faixa normal. A soma acumulada de alertas nunca diminui, por isso a busca binária pode ser feita sobre ela.

In [29]:
LIMITE = 5
linhas = []
for id_escada in ids_ordenados:
    alertas = [1 if esta_em_alerta(l) else 0 for l in por_escada[id_escada]]
    acumulados = acumular(alertas)
    pos_lin, c_lin = primeira_leitura_linear(alertas, LIMITE)
    pos_bin, c_bin = primeira_leitura_binaria(acumulados, LIMITE)
    assert pos_lin == pos_bin, f"As buscas divergem em {id_escada}"
    linhas.append((id_escada, pos_lin, c_lin, c_bin))

com_alerta = [x for x in linhas if x[1] != -1]
print(f"Escadas que chegam a {LIMITE} alertas: {len(com_alerta)} de {len(linhas)}")
print(f"Comparações: linear = {sum(x[2] for x in linhas)}, binária = {sum(x[3] for x in linhas)}")

Escadas que chegam a 5 alertas: 20 de 50
Comparações: linear = 6828, binária = 390


**Tabela por escada.** Para cada escada que chega a 5 alertas, mostramos a posição da primeira leitura (de 0 a 199 dentro da escada), o horário e as comparações feitas por cada busca. As duas buscas acham a mesma posição, como o `assert` acima confirma.

In [30]:
print(f"{'Escada':8} | {'Posição':>7} | {'Data e hora':16} | {'Comp. linear':>12} | {'Comp. binária':>13}")
for id_escada, pos, c_lin, c_bin in com_alerta:
    data = por_escada[id_escada][pos]["data_hora"]
    print(f"{id_escada:8} | {pos:>7} | {data:16} | {c_lin:>12} | {c_bin:>13}")


Escada   | Posição | Data e hora      | Comp. linear | Comp. binária
ESC-6    |      70 | 2026-01-03 22:00 |           71 |             8
ESC-11   |      31 | 2026-01-02 07:00 |           32 |             7
ESC-15   |      20 | 2026-01-01 20:00 |           21 |             8
ESC-16   |      25 | 2026-01-02 01:00 |           26 |             7
ESC-19   |      24 | 2026-01-02 00:00 |           25 |             8
ESC-21   |      19 | 2026-01-01 19:00 |           20 |             8
ESC-23   |      32 | 2026-01-02 08:00 |           33 |             8
ESC-24   |      75 | 2026-01-04 03:00 |           76 |             7
ESC-25   |      34 | 2026-01-02 10:00 |           35 |             7
ESC-26   |      23 | 2026-01-01 23:00 |           24 |             8
ESC-30   |      20 | 2026-01-01 20:00 |           21 |             8
ESC-31   |      84 | 2026-01-04 12:00 |           85 |             7
ESC-32   |      28 | 2026-01-02 04:00 |           29 |             7
ESC-33   |      35 | 2026-01-02 11

**Interpretação.** Vinte escadas chegam a 5 alertas, e as outras 30 não chegam. A busca binária faz bem menos comparações nesse cenário também. A diferença aparece porque a binária corta a busca ao meio a cada passo.

### Atividade 2: percurso na linha com pilha

**Passo 1.** Lemos a linha. A primeira estação é a base (Luz), e cada linha guarda os minutos até a estação seguinte.

In [31]:
from linha import ler_linha, tempo_ate, rota_ida_e_volta

linha = ler_linha("linha.csv")
base = linha[0]["estacao"]
print("Base:", base)
for registro in linha:
    print(f"  {registro['estacao']:14} | minutos até a seguinte: {registro['minutos_ate_proxima']}")

Base: Luz
  Luz            | minutos até a seguinte: 3
  República      | minutos até a seguinte: 3
  Paulista       | minutos até a seguinte: 2
  Oscar Freire   | minutos até a seguinte: 3
  Faria Lima     | minutos até a seguinte: 3
  Pinheiros      | minutos até a seguinte: 3
  Butantã        | minutos até a seguinte: 3
  Vila Sônia     | minutos até a seguinte: 0


**Passo 2.** `tempo_ate` acha a estação com busca linear e soma os minutos desde a base. A rota de ida e volta usa uma pilha: na ida, empilhamos com `append`; na volta, desempilhamos com `pop()`.

In [32]:
print(f"{'Estação':14} | {'ida (min)':>9} | {'ida e volta (min)':>17} | rota de volta")
for registro in linha:
    destino = registro["estacao"]
    rota = rota_ida_e_volta(linha, destino)
    print(f"{destino:14} | {tempo_ate(linha, destino):>9} | {rota['minutos_ida_volta']:>17} | {' -> '.join(rota['volta'])}")

Estação        | ida (min) | ida e volta (min) | rota de volta
Luz            |         0 |                 0 | Luz
República      |         3 |                 6 | República -> Luz
Paulista       |         6 |                12 | Paulista -> República -> Luz
Oscar Freire   |         8 |                16 | Oscar Freire -> Paulista -> República -> Luz
Faria Lima     |        11 |                22 | Faria Lima -> Oscar Freire -> Paulista -> República -> Luz
Pinheiros      |        14 |                28 | Pinheiros -> Faria Lima -> Oscar Freire -> Paulista -> República -> Luz
Butantã        |        17 |                34 | Butantã -> Pinheiros -> Faria Lima -> Oscar Freire -> Paulista -> República -> Luz
Vila Sônia     |        20 |                40 | Vila Sônia -> Butantã -> Pinheiros -> Faria Lima -> Oscar Freire -> Paulista -> República -> Luz


**Interpretação.** A volta sai sempre da estação de destino até a base, na ordem inversa da ida. Esse é o comportamento esperado de uma pilha: o último que entra é o primeiro que sai. Os tempos crescem em 3 minutos por estação, com exceção do trecho Paulista→Oscar Freire, que tem 2 minutos. Essas estimativas são premissas do grupo, não medições.

### Atividade 3: plano do turno com fila e pilha

**Passo 1.** Montamos as escadas em alerta. Para cada uma, guardamos a estação, o grupo e a data do primeiro alerta.

In [33]:
escadas_em_alerta = []
for id_escada in ids_ordenados:
    primeiro = None
    for leitura in por_escada[id_escada]:
        if esta_em_alerta(leitura):
            primeiro = leitura
            break
    if primeiro is not None:
        escadas_em_alerta.append({
            "id_escada": id_escada,
            "estacao": primeiro["estacao"],
            "grupo": primeiro["grupo"],
            "primeiro_alerta": primeiro["data_hora"],
        })
print("Escadas em alerta:", len(escadas_em_alerta))

Escadas em alerta: 20


**Passo 2.** Montamos o plano para 4, 6 e 8 horas. A fila crítica é atendida antes da normal (`pop(0)`). Cada escada custa ida + serviço + volta: 60 minutos de serviço para crítica e 40 para normal. Se não cabe no turno, a escada vai para pendentes.

In [34]:
from turno import planejar_turno, perder_hora

planos = {}
for horas in (4, 6, 8):
    plano = planejar_turno(base, horas, linha, escadas_em_alerta)
    planos[horas] = plano
    print(f"Turno de {horas} h: {len(plano['agenda'])} na agenda, "
          f"{len(plano['pendentes'])} pendentes, {plano['tempo_usado_min']} de {plano['limite_min']} min")

Turno de 4 h: 3 na agenda, 17 pendentes, 214 de 240 min
Turno de 6 h: 5 na agenda, 15 pendentes, 342 de 360 min
Turno de 8 h: 6 na agenda, 14 pendentes, 456 de 480 min


**Passo 3.** Simulamos a perda de 1 hora em cada turno. A agenda é desfeita com `pop()` até caber no novo limite.

In [35]:
for horas, plano in planos.items():
    depois = perder_hora(plano)
    print(f"{horas} h, se perder 1 hora: {len(depois['agenda'])} na agenda, "
          f"{len(depois['pendentes'])} pendentes, {depois['tempo_usado_min']} de {depois['limite_min']} min")

4 h, se perder 1 hora: 2 na agenda, 18 pendentes, 142 de 180 min
6 h, se perder 1 hora: 4 na agenda, 16 pendentes, 296 de 300 min
8 h, se perder 1 hora: 5 na agenda, 15 pendentes, 368 de 420 min


**Passo 4. Agenda do turno de 8 horas**, para ver a rota e os horários de cada atendimento.

In [36]:
print(f"{'Escada':8} | {'Grupo':12} | {'Estação':14} | {'Início':>6} | {'Fim':>5} | Rota de ida")
for item in planos[8]["agenda"]:
    print(f"{item['id_escada']:8} | {item['grupo']:12} | {item['estacao']:14} | "
          f"{item['inicio_min']:>6} | {item['fim_min']:>5} | {' -> '.join(item['rota_ida'])}")

Escada   | Grupo        | Estação        | Início |   Fim | Rota de ida
ESC-19   | crítica      | Faria Lima     |      0 |    82 | Luz -> República -> Paulista -> Oscar Freire -> Faria Lima
ESC-34   | crítica      | Luz            |     82 |   142 | Luz
ESC-24   | crítica      | Paulista       |    142 |   214 | Luz -> República -> Paulista
ESC-15   | crítica      | Faria Lima     |    214 |   296 | Luz -> República -> Paulista -> Oscar Freire -> Faria Lima
ESC-30   | crítica      | Paulista       |    296 |   368 | Luz -> República -> Paulista
ESC-16   | crítica      | Pinheiros      |    368 |   456 | Luz -> República -> Paulista -> Oscar Freire -> Faria Lima -> Pinheiros


**Interpretação.** Com 20 escadas em alerta e custo de 46 a 94 minutos cada, um turno de 8 horas cobre só 6. A maioria fica pendente em todos os cenários. Isso mostra que o problema não é a ordem de atendimento: é a capacidade do turno. As premissas de tempo de trecho e de serviço afetam diretamente esse resultado, então precisam ser validadas com a manutenção.

### Conclusão

1. **Etapa 0:** o gerador produz 50 escadas com 200 leituras cada, com semente 42. Os perfis são 30 saudáveis, 15 com desgaste e 5 com falha.
2. **Atividade 1:** a busca binária faz menos comparações que a linear. Para achar a primeira leitura com 5 alertas, a diferença é grande.
3. **Atividade 2:** a pilha dá a rota de ida e a volta na ordem inversa. Os tempos vêm das premissas do grupo.
4. **Atividade 3:** a fila crítica é atendida primeiro. O turno tem capacidade limitada, e a maioria das escadas fica pendente.

**Limitações:** os dados são fictícios; os tempos entre estações e de serviço são premissas; não há validação com falhas reais.